# Pico-Banana → SIGMA → scribble dataset (Colab)

Chọn **Runtime → Change runtime type → GPU**, rồi **Run all**. Mặc định upload ZIP cặp ảnh để tránh Flickr 429. Có thể chọn `DATA_SOURCE="remote"`
để tải trực tiếp. Không cần mount Drive.
Mặc định: 2 mẫu mỗi nhóm add/remove/replace/change, so sánh SIGMA hai ảnh và SIGMA hai ảnh + instruction,
3 box vẽ tay mỗi cặp. Chưa chạy fine-tune FLUX.2.

Nguồn: [Pico-Banana](https://github.com/apple-aiml-research/pico-banana-400k),
[SIGMA](https://github.com/ZhuangPeiyu/SIGMA).
Ảnh result là nhãn do editor tạo, có thể chứa lỗi. Mask ở đây là pseudo-label.
Không tính IoU/Dice khi chưa có mask thật. Dữ liệu gốc giữ các điều khoản sử dụng
của Pico-Banana/Open Images; file ZIP phục vụ thử nghiệm, không tự upload ra ngoài.

Notebook đã được kiểm tra cấu trúc/cú pháp và các hàm xử lý cục bộ;
chạy GPU Colab và tải checkpoint cần được xác nhận trên phiên Colab thực tế.


**Bản đa dạng scribble.** Bản notebook có output của lần chạy trước được giữ nguyên.
Mặc định xuất từ SIGMA visual theo đánh giá định tính 8 cặp; text vẫn được chạy để so sánh.
Sáu kiểu nét: box, ellipse, freehand_loop, cross, hatch, short_strokes.
Chỉ dùng cross cho remove; hatch/short_strokes cho remove/change. Add/replace dùng nét khoanh.


## 1. Cấu hình
`ROOT` có thể đổi sang thư mục Drive sau khi tự mount nếu muốn giữ cache qua các phiên.
`METHODS` hỗ trợ `sigma_visual`, `sigma_text`, `pixel_baseline`; không tự chuyển sang baseline khi SIGMA lỗi.
`MASK_THRESHOLD` là ngưỡng **xác suất mô hình**, không phải độ lệch RGB.
Thay mask/box không cần chạy lại model vì heatmap được cache riêng.


In [ ]:
from pathlib import Path
ROOT = Path("/content/pico_scribble_v2")
DATA_SOURCE = "upload"  # "upload" hoặc "remote"
UPLOAD_ZIP = ""  # đường dẫn ZIP có sẵn, hoặc để trống để chọn file trong Colab
METHODS = ["sigma_visual", "sigma_text", "pixel_baseline"]
N_PER_OPERATION = 2
SCAN_RECORDS = 6000  # lấy mẫu từ cửa sổ đầu metadata, không phải toàn dataset
CANDIDATES_PER_OPERATION = 24  # dự phòng URL Flickr hỏng
SEED = 42
INFER_SIZE = 518  # cấu hình inference upstream, chia hết cho patch size 14
MAX_IMAGE_SIDE = 1024
MAX_ASPECT_RATIO_ERROR = 0.10  # resize nhẹ + gắn cờ; >10% cần căn chỉnh trước
MASK_THRESHOLD = 0.5
PIXEL_THRESHOLD = 20 / 255
MIN_COMPONENT_FRACTION = 0.001
MAX_MASK_FRACTION = 0.65
BOX_PADDING = 0.08
N_VARIANTS = 6
EXPORT_METHOD = "sigma_visual"
SIGMA_REV = "29341a914344c6509ed7ce1c84fdf61c84258f64"
BASE_URL = "https://ml-site.cdn-apple.com/datasets/pico-banana-300k/nb/"
METADATA_URL = BASE_URL + "jsonl/sft.jsonl"
assert DATA_SOURCE in {"upload", "remote"}
assert set(METHODS) <= {"sigma_visual", "sigma_text", "pixel_baseline"}
assert EXPORT_METHOD in METHODS
assert INFER_SIZE % 14 == 0 and N_PER_OPERATION > 0
ROOT.mkdir(parents=True, exist_ok=True)

SCRIBBLE_STYLES = ["box", "ellipse", "freehand_loop", "cross", "hatch", "short_strokes"]
SCRIBBLE_VERSION = "v2-mixed-styles"


## 2. Dependency và GPU
Giữ PyTorch/CUDA của Colab. Nhánh text cần Qwen parser + GroundingDINO + SAM2.1.
Cài LangSAM/SAM2 đúng commit upstream với `--no-deps` để không thay PyTorch.
Nếu đã chạy bản notebook cũ và gặp xung đột import, bắt đầu một runtime mới.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "timm==1.0.25", "huggingface_hub>=0.34,<2", "PyYAML>=6,<7",
                "requests>=2.32,<3", "pillow>=10", "matplotlib>=3.8",
                "opencv-python-headless>=4.10,<5", "ipywidgets>=8,<9", "transformers>=4.49,<5", "accelerate>=1,<2",
                "sentencepiece", "scipy", "hydra-core>=1.3,<2", "iopath", "supervision>=0.23,<0.28"], check=True)
import torch
print("Python:", sys.version, "Torch:", torch.__version__)
if any(m.startswith("sigma_") for m in METHODS):
    assert torch.cuda.is_available(), "Bật GPU runtime để chạy SIGMA."
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(), "VRAM GiB:",
          round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))


if "sigma_text" in METHODS:
    import os
    env = dict(os.environ, SAM2_BUILD_CUDA="0")
    for package in (
        "git+https://github.com/facebookresearch/segment-anything-2@c2ec8e14a185632b0a5d8b161928ceb50197eddc",
        "git+https://github.com/luca-medeiros/lang-segment-anything@918043ed4666eea04da88aa179eb8d27ef4b1a1d",
    ):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                        "--no-build-isolation", package], check=True, env=env)


In [ ]:
import csv, hashlib, io, json, random, shutil, time, warnings, zipfile
from pathlib import PurePosixPath
from collections import Counter
from urllib.parse import urljoin, urlparse
import cv2
import numpy as np
import requests
from PIL import Image, ImageDraw, ImageOps
import matplotlib.pyplot as plt
from IPython.display import display, clear_output, FileLink

session = requests.Session()
session.headers["User-Agent"] = "Mozilla/5.0 (Pico-Scribble research notebook)"

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    temp.replace(path)

def operation_of(edit_type):
    t = str(edit_type).lower()
    if "relocat" in t or t.startswith("move "):
        return None
    if t.startswith("add a new object"):
        return "add"
    if t.startswith("remove "):
        return "remove"
    if t.startswith("replace "):
        return "replace"
    if "overall" not in t and any(x in t for x in ("color", "material", "texture")):
        return "change"
    return None

def normalize_record(raw):
    # Schema thật của sft.jsonl: output_image là đường dẫn tương đối.
    source = raw.get("open_image_input_url", "")
    target = urljoin(BASE_URL, raw.get("output_image", ""))
    prompt = str(raw.get("text", "")).strip()
    op = operation_of(raw.get("edit_type", ""))
    if not op or not source or not raw.get("output_image") or not prompt:
        return None
    if any(urlparse(u).scheme not in ("http", "https") for u in (source, target)):
        return None
    uid = hashlib.sha256((source + "|" + target).encode()).hexdigest()[:16]
    return dict(id=uid, operation=op, edit_type=raw["edit_type"],
                prompt_original=prompt, prompt_short=str(raw.get("summarized_text") or "").strip(),
                source_url=source, result_url=target)

def fetch_image(url, path):
    path = Path(path)
    if path.exists():
        with Image.open(path) as image:
            image.verify()
        return
    error = None
    for attempt in range(3):
        try:
            with session.get(url, timeout=(15, 45), stream=True) as response:
                if response.status_code == 429:
                    raise RuntimeError(f"HTTP 429 rate limited: {url}. Dùng DATA_SOURCE=upload.")
                response.raise_for_status()
                payload = bytearray()
                for chunk in response.iter_content(65536):
                    payload.extend(chunk)
                    if len(payload) > 40 * 1024**2:
                        raise ValueError("Image exceeds 40 MiB")
            with Image.open(io.BytesIO(payload)) as image:
                image = ImageOps.exif_transpose(image).convert("RGB")
                path.parent.mkdir(parents=True, exist_ok=True)
                image.save(path, format="PNG")
            return
        except (requests.RequestException, OSError, ValueError) as exc:
            error = exc
            time.sleep(attempt + 1)
    raise RuntimeError(f"Download failed: {url}: {error}")


def load_uploaded_pairs(zip_path, root):
    """Accept a Prior_test ZIP or a pairs.jsonl ZIP with local image paths."""
    digest = hashlib.sha256(Path(zip_path).read_bytes()).hexdigest()[:16]
    target = root / "uploaded" / digest
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        members = archive.infolist()
        if sum(m.file_size for m in members) > 2 * 1024**3:
            raise ValueError("ZIP uncompressed size exceeds 2 GiB")
        for member in members:
            rel = PurePosixPath(member.filename.replace("\\", "/"))
            if rel.is_absolute() or ".." in rel.parts or any(":" in part for part in rel.parts):
                raise ValueError(f"Unsafe ZIP path: {member.filename}")
            if member.is_dir():
                continue
            dest = (target / Path(*rel.parts)).resolve()
            if not dest.is_relative_to(target.resolve()):
                raise ValueError("ZIP path escapes destination")
            dest.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, dest.open("wb") as output:
                shutil.copyfileobj(source, output)

    candidates = []
    # Repo's data/Prior_test/<operation>/<case>/metadata.json
    for metadata in sorted(target.rglob("metadata.json")):
        obj = json.loads(metadata.read_text(encoding="utf-8"))
        if isinstance(obj, dict) and isinstance(obj.get("samples"), list):
            for row in obj["samples"]:
                candidates.append((metadata.parent, dict(row, operation=obj.get("operation", "").lower())))
    # Custom ZIP layout: pairs.jsonl + original/... + result/...
    for manifest in sorted(target.rglob("pairs.jsonl")):
        for line in manifest.read_text(encoding="utf-8").splitlines():
            if line.strip():
                candidates.append((manifest.parent, json.loads(line)))
    if not candidates:
        raise ValueError("ZIP cần có pairs.jsonl hoặc metadata.json của Prior_test.")
    selected, errors, counts = [], [], Counter()
    random.Random(SEED).shuffle(candidates)
    for base, row in candidates:
        op = str(row.get("operation", "")).lower()
        if op not in {"add", "remove", "replace", "change"} or counts[op] >= N_PER_OPERATION:
            continue
        try:
            source_name = row.get("original_file_name") or row.get("image")
            result_name = row.get("result_file_name") or row.get("ground_truth")
            if not source_name or not result_name:
                raise ValueError("Missing original/result path")
            paths = [(base / name).resolve() for name in (source_name, result_name)]
            if not all(path.is_relative_to(target.resolve()) for path in paths):
                raise ValueError("Image path escapes uploaded directory")
            source_hash = hashlib.sha256(paths[0].read_bytes()).hexdigest()
            uid = hashlib.sha256(paths[0].read_bytes() + paths[1].read_bytes()).hexdigest()[:16]
            with Image.open(paths[0]) as aa, Image.open(paths[1]) as bb:
                a, b, alignment = prepare_images(ImageOps.exif_transpose(aa), ImageOps.exif_transpose(bb))
                for folder, image in (("original", a), ("result", b)):
                    (root / folder).mkdir(exist_ok=True)
                    image.save(root / folder / f"{uid}.png")
            prompt = str(row.get("prompt_original") or row.get("text") or "").strip()
            if not prompt:
                raise ValueError("Missing instruction text")
            rec = dict(id=uid, operation=op, edit_type=row.get("edit_type", op),
                       prompt_original=prompt, prompt_short=str(row.get("prompt_short") or "").strip(),
                       source_url="upload:sha256:" + source_hash,
                       result_url=row.get("output_image") or "upload:" + str(result_name), **alignment)
            if any(p["id"] == uid for p in selected):
                continue
            selected.append(rec)
            counts[op] += 1
        except Exception as exc:
            errors.append(dict(record=row, error=str(exc)))
            print("Upload pair skipped:", exc)
    write_json(root / "upload_errors.json", errors)
    if not selected:
        raise ValueError("No valid uploaded pairs; see upload_errors.json")
    return selected, dict(source="upload", zip_sha256=digest, seed=SEED,
                           n=N_PER_OPERATION, max_side=MAX_IMAGE_SIDE, max_aspect_ratio_error=MAX_ASPECT_RATIO_ERROR)


def prepare_images(a, b):
    source_size, result_size = list(a.size), list(b.size)
    delta = abs((a.width / a.height) / (b.width / b.height) - 1)
    if delta > MAX_ASPECT_RATIO_ERROR:
        raise ValueError(f"Aspect ratio error {delta:.1%} > {MAX_ASPECT_RATIO_ERROR:.1%}; align/crop first")
    a = a.convert("RGB")
    a.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE), Image.Resampling.LANCZOS)
    b = b.convert("RGB").resize(a.size, Image.Resampling.LANCZOS)
    alignment = dict(source_size=source_size, result_size=result_size, aspect_ratio_error=delta,
                     alignment_flags=["aspect_ratio_resized"] if delta > 0.02 else [])
    return a, b, alignment


import logging
class GroundingLogCapture(logging.Handler):
    def __init__(self):
        super().__init__()
        self.failures = []
    def emit(self, record):
        message = record.getMessage()
        if "LangSAM strict prob failed" in message:
            self.failures.append(message)


## 3. Nạp ảnh: upload ZIP hoặc tải trực tiếp
**Upload (mặc định):** chọn ZIP chứa `Prior_test/.../metadata.json` và các ảnh tương ứng,
hoặc ZIP tùy chỉnh có cấu trúc:

```text
pairs.jsonl
original/a.png
result/a.png
```

Một dòng `pairs.jsonl`:
```json
{"operation":"remove","prompt_original":"Remove the red flag.","prompt_short":"Remove the flag.","original_file_name":"original/a.png","result_file_name":"result/a.png"}
```

Không upload notebook/output của SIGMA thay cho cặp ảnh. Instruction gốc bắt buộc;
prompt ngắn có thể bổ sung ở gallery. Upload bỏ qua Flickr hoàn toàn.
Chế độ remote hiển thị lỗi cụ thể và ghi log ngay sau từng lỗi, kể cả khi chưa tải được cặp nào.
Tăng SCAN_RECORDS không giải quyết rate limit HTTP 429.

Lệch tỷ lệ ≤10%: resize theo ảnh gốc; lệch >2% gắn cờ `aspect_ratio_resized`.
Đây không phải căn chỉnh hình học, hãy kiểm tra vùng biên trong gallery.


In [ ]:

if DATA_SOURCE == "upload":
    if UPLOAD_ZIP:
        zip_path = Path(UPLOAD_ZIP)
    else:
        from google.colab import files
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
        assert len(zip_names) == 1, "Chọn đúng một file ZIP."
        zip_path = Path(zip_names[0])
    pairs, selection_config = load_uploaded_pairs(zip_path, ROOT)
    write_json(ROOT / "uploaded_pairs.json", dict(selection_config=selection_config, pairs=pairs))
    print("Uploaded:", dict(Counter(p["operation"] for p in pairs)))
else:
    manifest_path = ROOT / "pairs.json"
    selection_config = dict(seed=SEED, n=N_PER_OPERATION, scan=SCAN_RECORDS,
                            candidates=CANDIDATES_PER_OPERATION, max_side=MAX_IMAGE_SIDE,
                            metadata_url=METADATA_URL, max_aspect_ratio_error=MAX_ASPECT_RATIO_ERROR)
    if manifest_path.exists():
        saved = json.loads(manifest_path.read_text())
        assert saved["selection_config"] == selection_config, "Đổi ROOT khi đổi cấu hình lấy mẫu."
        pairs = saved["pairs"]
        assert pairs, "Cache rỗng; đổi ROOT hoặc dùng upload."
    else:
        buckets = {op: [] for op in ("add", "remove", "replace", "change")}
        seen = Counter()
        rng = random.Random(SEED)
        with session.get(METADATA_URL, stream=True, timeout=(15, 60)) as response:
            response.raise_for_status()
            for index, line in enumerate(response.iter_lines()):
                if index >= SCAN_RECORDS:
                    break
                if not line:
                    continue
                rec = normalize_record(json.loads(line))
                if rec is None:
                    continue
                op = rec["operation"]
                seen[op] += 1
                bucket = buckets[op]
                if len(bucket) < CANDIDATES_PER_OPERATION:
                    bucket.append(rec)
                else:
                    j = rng.randrange(seen[op])
                    if j < CANDIDATES_PER_OPERATION:
                        bucket[j] = rec
        print("Eligible records scanned:", dict(seen))
        pairs, errors = [], []
        for op, candidates in buckets.items():
            rng.shuffle(candidates)
            kept = 0
            for rec in candidates:
                if kept >= N_PER_OPERATION:
                    break
                uid = rec["id"]
                raw_a, raw_b = ROOT / "raw" / f"{uid}_a.png", ROOT / "raw" / f"{uid}_b.png"
                try:
                    fetch_image(rec["source_url"], raw_a)
                    fetch_image(rec["result_url"], raw_b)
                    with Image.open(raw_a) as a, Image.open(raw_b) as b:
                        a, b, alignment = prepare_images(a, b)
                        rec.update(alignment)
                        for folder, image in (("original", a), ("result", b)):
                            out = ROOT / folder / f"{uid}.png"
                            out.parent.mkdir(exist_ok=True)
                            image.save(out)
                    pairs.append(rec)
                    kept += 1
                    print(op, kept, uid)
                except Exception as exc:
                    errors.append(dict(id=uid, source_url=rec["source_url"], result_url=rec["result_url"], error=str(exc)))
                    write_json(ROOT / "download_errors.json", errors)
                    print(f"SKIP {op} {uid}: {exc}")
                    if "429" in str(exc):
                        raise RuntimeError("Flickr rate limit; chọn DATA_SOURCE=upload. Log: " + str(ROOT / "download_errors.json")) from exc
            if kept < N_PER_OPERATION:
                warnings.warn(f"{op}: chỉ tải được {kept}/{N_PER_OPERATION}; xem download_errors.json hoặc dùng upload.")
        write_json(ROOT / "download_errors.json", errors)
        assert pairs, "Không tải được cặp ảnh; xem download_errors.json."
        write_json(manifest_path, dict(selection_config=selection_config, pairs=pairs))
    print("Loaded:", dict(Counter(p["operation"] for p in pairs)))


## 4. Load SIGMA và tài nguyên nhánh text
Hai chế độ dùng chung checkpoint: `sigma_visual` không truyền instruction;
`sigma_text` truyền **instruction gốc đầy đủ**, chạy Qwen → GroundingDINO/SAM2.1 → fusion.
Tải tài nguyên text chỉ khi có sigma_text. Giữ external modules lazy để checkpoint
SIGMA không bị lẫn tham số LLM/detector khi strict load.


In [ ]:
model = None
checkpoint_revision = None
if any(m.startswith("sigma_") for m in METHODS):
    from huggingface_hub import HfApi, hf_hub_download, snapshot_download
    repo = ROOT / "SIGMA"
    if not repo.exists():
        subprocess.run(["git", "clone", "https://github.com/ZhuangPeiyu/SIGMA.git", str(repo)], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", "--detach", SIGMA_REV], check=True)
    sys.path.insert(0, str(repo / "src"))
    from highaccmaskgen.sigma import build_ffd_net, load_checkpoint
    checkpoint_revision = HfApi().model_info("PatrickZZZ/SIGMA").sha
    checkpoint = hf_hub_download("PatrickZZZ/SIGMA", "model.pt", revision=checkpoint_revision)
    patterns = ["resource/models--timm--vit_base_patch14_dinov2.lvd142m/**"]
    if "sigma_text" in METHODS:
        patterns += ["resource/models--Qwen--Qwen2.5-0.5B-Instruct/**",
                     "resource/models--IDEA-Research--grounding-dino-base/**",
                     "resource/sam2.1_hiera_small.pt"]
    snapshot_download("PatrickZZZ/SIGMA", revision=checkpoint_revision,
                      allow_patterns=patterns,
                      local_dir=repo)
    resource = repo / "resource"
    text_config = {}
    if "sigma_text" in METHODS:
        gdino_snapshots = sorted((resource / "models--IDEA-Research--grounding-dino-base" / "snapshots").glob("*/config.json"))
        assert gdino_snapshots, "Missing GroundingDINO snapshot"
        text_config = dict(parser_model_path=str(resource / "models--Qwen--Qwen2.5-0.5B-Instruct"),
                           parser_torch_dtype="float16", parser_device_map="auto",
                           langsam_sam_ckpt_path=str(resource / "sam2.1_hiera_small.pt"),
                           langsam_gdino_model_ckpt_path=str(gdino_snapshots[0].parent),
                           langsam_gdino_processor_ckpt_path=str(gdino_snapshots[0].parent))
    model = build_ffd_net(dict(
        ffd_preset="ffd_msdm_drb2_textbranch",
        encoder_name="vit_base_patch14_dinov2.lvd142m",
        encoder_trainable=False, hf_hub_cache=str(repo / "resource"),
        enable_online_text_modules=False, **text_config), encoder_pretrained=True)
    stats = load_checkpoint(model, Path(checkpoint), device="cpu", strict=True)
    print(stats)
    model = model.eval().to("cuda")
    print("SIGMA ready; checkpoint revision:", checkpoint_revision)


## 5. Inference một cặp trước, rồi chạy các cặp còn lại
Heatmap SIGMA ở thang xác suất; baseline là độ lệch RGB trung bình sau làm mờ nhẹ.
Mỗi phương pháp có cache riêng, cache key chứa hash ảnh + cấu hình model/checkpoint.
Lỗi inference làm dừng cell để xem nguyên nhân, không được gắn nhãn thành công giả.


In [ ]:
def read_pair(rec):
    a = np.array(Image.open(ROOT / "original" / f"{rec['id']}.png").convert("RGB"))
    b = np.array(Image.open(ROOT / "result" / f"{rec['id']}.png").convert("RGB"))
    assert a.shape == b.shape
    return a, b

def predict_pair(rec, method):
    a, b = read_pair(rec)
    signature = hashlib.sha256(a.tobytes() + b.tobytes() + json.dumps(dict(
        method=method, infer_size=INFER_SIZE, sigma_rev=SIGMA_REV,
        checkpoint_revision=checkpoint_revision if method.startswith("sigma_") else None,
        instruction=rec["prompt_original"] if method == "sigma_text" else None
    ), sort_keys=True).encode()).hexdigest()[:20]
    path = ROOT / "heatmaps" / method / f"{rec['id']}_{signature}.npy"
    if path.exists():
        return np.load(path)
    started = time.perf_counter()
    if method in {"sigma_visual", "sigma_text"}:
        assert model is not None
        import torch.nn.functional as F
        def tensor(image):
            image = cv2.resize(image, (INFER_SIZE, INFER_SIZE), interpolation=cv2.INTER_AREA)
            return torch.from_numpy(image).permute(2, 0, 1).float().unsqueeze(0).cuda() / 255
        with torch.inference_mode():
            kwargs = {"instructions": [rec["prompt_original"]]} if method == "sigma_text" else {}
            capture = GroundingLogCapture()
            logger = logging.getLogger()
            logger.addHandler(capture)
            try:
                output = model(tensor(a), tensor(b), **kwargs)
            finally:
                logger.removeHandler(capture)
            if capture.failures:
                raise RuntimeError("Text grounding failed; no valid comparison: " + " | ".join(capture.failures))
            prob = F.interpolate(output["mask_logits"].sigmoid().float(),
                                 size=a.shape[:2], mode="bilinear", align_corners=False)
            heat = prob[0, 0].cpu().numpy()
    elif method == "pixel_baseline":
        aa = cv2.GaussianBlur(a.astype(np.float32) / 255, (5, 5), 0)
        bb = cv2.GaussianBlur(b.astype(np.float32) / 255, (5, 5), 0)
        heat = np.abs(aa - bb).mean(axis=2)
    else:
        raise ValueError(method)
    assert heat.shape == a.shape[:2] and np.isfinite(heat).all()
    if method == "sigma_text" and model.text_branch.semantic_parser is None:
        raise RuntimeError("Text branch did not initialize; do not label this sigma_text.")
    path.parent.mkdir(parents=True, exist_ok=True)
    np.save(path, heat.astype(np.float32))
    write_json(path.with_suffix(".json"), dict(seconds=time.perf_counter() - started,
               method=method, signature=signature, source_pair_id=rec["id"]))
    return heat

for method in METHODS:
    heat = predict_pair(pairs[0], method)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    a, b = read_pair(pairs[0])
    for ax, image, title in zip(axes, (a, b, heat), ("Original", "Result", method)):
        ax.imshow(image, cmap="magma", vmin=0, vmax=1) if image.ndim == 2 else ax.imshow(image)
        ax.set_title(title); ax.axis("off")
    plt.show()


In [ ]:
for rec in pairs:
    for method in METHODS:
        predict_pair(rec, method)
    print("Done:", rec["operation"], rec["id"])
if torch.cuda.is_available():
    print("Peak allocated VRAM GiB:", round(torch.cuda.max_memory_allocated() / 2**30, 2))


# So sánh hai heatmap trên cùng cặp, cùng scale; không dùng IoU nếu chưa có GT mask.
if {"sigma_visual", "sigma_text"} <= set(METHODS):
    for rec in pairs:
        a, b = read_pair(rec)
        fig, axes = plt.subplots(1, 4, figsize=(18, 4))
        for ax, image, title in zip(axes, (a, b, predict_pair(rec, "sigma_visual"),
                                          predict_pair(rec, "sigma_text")),
                                   ("Original", "Result", "SIGMA visual", "SIGMA text")):
            if image.ndim == 2:
                ax.imshow(image, cmap="magma", vmin=0, vmax=1)
            else:
                ax.imshow(image)
            ax.set_title(title); ax.axis("off")
        fig.suptitle(rec["prompt_original"], wrap=True, fontsize=9)
        plt.show()


## 6. Mask → box → scribble
Giữ các component đủ lớn; không tự lấy mỗi vùng lớn nhất vì có thể mất vật thể nhỏ.
Box dùng tọa độ `[x0,y0,x1,y1)` trong ảnh original đã chuẩn hóa.
Các mask trống/quá rộng được gắn cờ để duyệt. Không dùng mask như một nhãn chính xác.
Chỉ tạo box vẽ tay ở bản này; relocation và cue kiểu mũi tên cần pipeline riêng.


In [ ]:
def mask_and_boxes(heat, threshold, min_fraction=MIN_COMPONENT_FRACTION, padding=BOX_PADDING):
    binary = (heat >= threshold).astype(np.uint8)
    h, w = binary.shape
    n, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    mask = np.zeros_like(binary)
    boxes = []
    for label in range(1, n):
        x, y, bw, bh, area = map(int, stats[label])
        if area < max(4, int(h * w * min_fraction)):
            continue
        mask[labels == label] = 255
        px, py = max(2, round(bw * padding)), max(2, round(bh * padding))
        boxes.append([max(0, x-px), max(0, y-py), min(w, x+bw+px), min(h, y+bh+py)])
    boxes.sort(key=lambda b: (b[1], b[0]))
    flags = []
    fraction = float((mask > 0).mean())
    if not boxes:
        flags.append("empty_mask")
    if fraction > MAX_MASK_FRACTION:
        flags.append("large_mask")
    if len(boxes) > 8:
        flags.append("many_components")
    return mask, boxes, flags

def allowed_styles(operation):
    common = ["box", "ellipse", "freehand_loop"]
    if operation == "remove":
        return common + ["cross", "hatch", "short_strokes"]
    if operation == "change":
        return common + ["hatch", "short_strokes"]
    return common

def style_for(rec, variant):
    choices = [s for s in allowed_styles(rec["operation"]) if s in SCRIBBLE_STYLES]
    if not choices:
        raise ValueError("No suitable scribble styles for operation")
    random.Random(SEED + int(rec["id"][:8], 16)).shuffle(choices)
    return choices[variant % len(choices)]

def draw_scribble(image, boxes, seed, style="box", mask=None):
    if style not in {"box", "ellipse", "freehand_loop", "cross", "hatch", "short_strokes"}:
        raise ValueError(style)
    if style in {"hatch", "short_strokes"} and mask is None:
        raise ValueError("Mask required for interior scribble strokes")
    rng = np.random.default_rng(seed)
    base = Image.fromarray(image).convert("RGBA")
    layer = Image.new("RGBA", base.size)
    draw = ImageDraw.Draw(layer)
    color = [(255, 30, 30), (20, 210, 70), (20, 100, 255)][int(rng.integers(3))]
    rgba = (*color, int(rng.integers(190, 256)))
    for x0, y0, x1, y1 in boxes:
        bw, bh = x1-x0, y1-y0
        cx, cy = (x0+x1-1)/2, (y0+y1-1)/2
        width = max(1, min(round(min(base.size)*rng.uniform(.003, .007)), round(min(bw,bh)*.06)))
        jitter = max(.5, min(bw,bh)*rng.uniform(.01, .04))
        def line(points):
            points = [(round(np.clip(x,0,base.width-1)), round(np.clip(y,0,base.height-1))) for x,y in points]
            draw.line(points, fill=rgba, width=width, joint="curve")
        if style == "box":
            corners = [(x0,y0), (x1-1,y0), (x1-1,y1-1), (x0,y1-1), (x0,y0)]
            points = []
            for start,end in zip(corners,corners[1:]):
                knots = rng.normal(0,jitter,(5,2))
                for t in np.linspace(0,1,24,endpoint=False):
                    delta = [np.interp(t,np.linspace(0,1,5),knots[:,d]) for d in (0,1)]
                    points.append((1-t)*np.array(start)+t*np.array(end)+delta)
            line(points[int(rng.integers(0,6)):])
        elif style in {"ellipse", "freehand_loop"}:
            angle = rng.uniform(-.15,.15)
            phases = rng.uniform(0,2*np.pi,2)
            points = []
            # Enclosing ellipse/loop derives from a coarse box, not the exact result contour.
            for t in np.linspace(0,2*np.pi-rng.uniform(0,.18),120):
                radius = 1 if style == "ellipse" else 1+.06*np.sin(3*t+phases[0])+.035*np.sin(5*t+phases[1])
                x,y = bw*.55*radius*np.cos(t), bh*.55*radius*np.sin(t)
                points.append((cx+x*np.cos(angle)-y*np.sin(angle), cy+x*np.sin(angle)+y*np.cos(angle)))
            line(points)
        elif style == "cross":
            for start,end in (((x0,y0),(x1-1,y1-1)),((x0,y1-1),(x1-1,y0))):
                points = []
                for t in np.linspace(0,1,24):
                    points.append((1-t)*np.array(start)+t*np.array(end)+rng.normal(0,jitter,2))
                line(points)
        elif style == "hatch":
            step = max(width*4, min(bw,bh)*rng.uniform(.15,.25))
            for offset in np.arange(-bh,bw,step):
                line([(x0+offset,y1-1),(x0+offset+bh,y0)])
        else:
            ys,xs = np.where(mask[y0:y1,x0:x1]>0)
            if not len(xs):
                continue
            for _ in range(int(rng.integers(3,7))):
                idx = int(rng.integers(len(xs)))
                center = np.array([x0+xs[idx],y0+ys[idx]])
                angle = rng.uniform(0,2*np.pi)
                direction = np.array([np.cos(angle),np.sin(angle)])
                length = min(bw,bh)*rng.uniform(.2,.5)
                line([center+t*length*direction+rng.normal(0,jitter,2) for t in np.linspace(-.5,.5,16)])
    if style in {"hatch", "short_strokes"}:
        binary = (mask>0).astype(np.uint8)*255
        h,w = binary.shape
        assert (w,h) == base.size
        kernel = max(1, round(min(base.size)*.003))*2+1
        interior = cv2.erode(binary,np.ones((kernel,kernel),np.uint8))
        # Preserve very small components that vanish under erosion.
        count, labels = cv2.connectedComponents(binary)
        for label in range(1,count):
            component = labels == label
            if not np.any(interior[component]):
                interior[component] = 255
        alpha = np.asarray(layer.getchannel("A"),dtype=np.uint8)
        layer.putalpha(Image.fromarray(np.minimum(alpha,interior)))
    return Image.alpha_composite(base,layer).convert("RGB")

# Offline smoke checks for all styles: deterministic, visible, same image dimensions.
_heat = np.zeros((160,240),np.float32)
_heat[30:100,25:95] = 1; _heat[50:140,150:220] = 1
_mask,_boxes,_flags = mask_and_boxes(_heat,.5)
assert len(_boxes)==2 and not _flags
_image = np.zeros((160,240,3),np.uint8)
for _style in ("box","ellipse","freehand_loop","cross","hatch","short_strokes"):
    _a = np.array(draw_scribble(_image,_boxes,42,_style,_mask))
    _b = np.array(draw_scribble(_image,_boxes,42,_style,_mask))
    assert _a.shape==_image.shape and np.any(_a!=_image) and np.array_equal(_a,_b)
    if _style in {"hatch","short_strokes"}:
        assert not np.any(_a[_mask==0])
assert "cross" not in allowed_styles("add")
print("All six scribble styles passed offline checks")


## Xem các kiểu nét trên một mẫu
Ví dụ minh họa dùng mẫu remove để cả sáu kiểu đều có ý nghĩa.
Nét cross dành cho xóa. Các nét trong mask được dùng cho remove/change;
không dùng contour đích chính xác để gợi hình vật thể cần thêm.


In [ ]:

example = next((p for p in pairs if p["operation"]=="remove"),pairs[0])
heat = predict_pair(example,EXPORT_METHOD)
mask,boxes,flags = mask_and_boxes(heat,MASK_THRESHOLD)
a,b = read_pair(example)
styles = allowed_styles(example["operation"])
fig,axes = plt.subplots(2,3,figsize=(15,9))
for ax in axes.flat:
    ax.axis("off")
for ax,style in zip(axes.flat,styles):
    ax.imshow(draw_scribble(a,boxes,SEED,style,mask)); ax.set_title(style)
plt.tight_layout(); plt.show()


## 7. Gallery và duyệt từng cặp
Chọn method/sample; sửa threshold để xem lại mask mà không chạy model.
Sửa prompt ngắn từ `summarized_text` có sẵn của Pico-Banana, tránh tự cắt cụt instruction.
Nếu metadata thiếu prompt ngắn, cần nhập trước khi duyệt. Chỉ đánh dấu **approved**
khi vùng, prompt và result đều hợp lý. Notebook không tự suy ra chất lượng chỉ từ diện tích mask.


In [ ]:
import ipywidgets as widgets
reviews_path = ROOT / "reviews.json"
reviews = json.loads(reviews_path.read_text()) if reviews_path.exists() else {}
sample_widget = widgets.Dropdown(options=[(f"{p['operation']} / {p['id']}", i) for i, p in enumerate(pairs)])
method_widget = widgets.Dropdown(options=METHODS, value=EXPORT_METHOD)
threshold_widget = widgets.FloatSlider(min=0.001, max=1, step=0.005, value=MASK_THRESHOLD, description="Threshold")
prompt_widget = widgets.Textarea(description="Short prompt", layout=widgets.Layout(width="95%"))
status_widget = widgets.Dropdown(options=["pending", "approved", "rejected"])
note_widget = widgets.Text(description="Note", layout=widgets.Layout(width="95%"))
style_widget = widgets.Dropdown(options=allowed_styles(pairs[0]["operation"]), description="Scribble")
save_button = widgets.Button(description="Lưu duyệt", button_style="success")
view = widgets.Output()

def review_key(rec, method):
    return method + ":" + rec["id"]

def show_preview(*_):
    rec, method = pairs[sample_widget.value], method_widget.value
    a, b = read_pair(rec)
    heat = predict_pair(rec, method)
    mask, boxes, flags = mask_and_boxes(heat, threshold_widget.value)
    box_image = Image.fromarray(a.copy())
    drawing = ImageDraw.Draw(box_image)
    for x0, y0, x1, y1 in boxes:
        drawing.rectangle((x0, y0, x1-1, y1-1), outline="red", width=3)
    overlay = draw_scribble(a, boxes, SEED, style_widget.value, mask)
    with view:
        clear_output(wait=True)
        print(rec["prompt_original"])
        print("Flags:", rec.get("alignment_flags", []) + flags, "mask fraction:", round(float((mask > 0).mean()), 3))
        fig, axes = plt.subplots(2, 3, figsize=(15, 9))
        for ax, image, title in zip(axes.flat, (a, b, heat, mask, box_image, overlay),
                                   ("Original", "Result", "Heatmap", "Mask", "Boxes", "Scribble")):
            if isinstance(image, np.ndarray) and image.ndim == 2:
                ax.imshow(image, cmap="magma" if title == "Heatmap" else "gray")
            else:
                ax.imshow(image)
            ax.set_title(title); ax.axis("off")
        plt.tight_layout(); plt.show()

def select_sample(*_):
    rec, method = pairs[sample_widget.value], method_widget.value
    style_widget.options = allowed_styles(rec["operation"])
    review = reviews.get(review_key(rec, method), {})
    threshold_widget.value = review.get("threshold", PIXEL_THRESHOLD if method == "pixel_baseline" else MASK_THRESHOLD)
    prompt_widget.value = review.get("prompt_short", rec["prompt_short"])
    status_widget.value = review.get("status", "pending")
    note_widget.value = review.get("note", "")
    show_preview()

def save_review(_):
    rec, method = pairs[sample_widget.value], method_widget.value
    mask, boxes, flags = mask_and_boxes(predict_pair(rec, method), threshold_widget.value)
    if status_widget.value == "approved" and (not prompt_widget.value.strip() or not boxes):
        with view:
            print("Không thể approve khi prompt trống hoặc không có box.")
        return
    reviews[review_key(rec, method)] = dict(status=status_widget.value,
        prompt_short=prompt_widget.value.strip(), threshold=threshold_widget.value,
        note=note_widget.value, flags=flags)
    write_json(reviews_path, reviews)
    with view:
        print("Saved", rec["id"], status_widget.value)

sample_widget.observe(select_sample, names="value")
method_widget.observe(select_sample, names="value")
threshold_widget.observe(show_preview, names="value")
style_widget.observe(show_preview, names="value")
save_button.on_click(save_review)
display(widgets.VBox([widgets.HBox([sample_widget, method_widget]), style_widget, threshold_widget,
                      prompt_widget, widgets.HBox([status_widget, save_button]), note_widget, view]))
select_sample()


## 8. Xuất dataset và ZIP
Run all xuất bản **preview** (các mẫu còn pending), không tự coi là dữ liệu đã duyệt.
Sau khi duyệt ở cell trên, đặt `APPROVED_ONLY=True` và chạy lại cell này để xuất tập train.
Split theo hash **source URL** trước khi tạo biến thể để ảnh gốc trùng không lọt qua các split.
Với 8 mẫu split chỉ minh họa, chưa đủ để đánh giá fine-tune.
File ảnh gốc trong bản xuất là ảnh sạch đã xử lý EXIF/resize; file tải về được giữ ở `raw/`.


In [ ]:
APPROVED_ONLY = False
export_config = dict(method=EXPORT_METHOD, approved_only=APPROVED_ONLY, seed=SEED,
    variants=N_VARIANTS, threshold=MASK_THRESHOLD, pixel_threshold=PIXEL_THRESHOLD,
    padding=BOX_PADDING, min_component_fraction=MIN_COMPONENT_FRACTION,
    sigma_rev=SIGMA_REV, checkpoint_revision=checkpoint_revision,
    scribble_version=SCRIBBLE_VERSION, scribble_styles=SCRIBBLE_STYLES)
tag = hashlib.sha256(json.dumps(dict(config=export_config, reviews=reviews),
                                    sort_keys=True).encode()).hexdigest()[:12]
export_dir = ROOT / "exports" / tag
export_dir.mkdir(parents=True, exist_ok=True)
rows, review_rows = [], []
for rec in pairs:
    review = reviews.get(review_key(rec, EXPORT_METHOD), {})
    status = review.get("status", "pending")
    if status == "rejected" or (APPROVED_ONLY and status != "approved"):
        continue
    threshold = review.get("threshold", PIXEL_THRESHOLD if EXPORT_METHOD == "pixel_baseline" else MASK_THRESHOLD)
    a, b = read_pair(rec)
    heat = predict_pair(rec, EXPORT_METHOD)
    mask, boxes, flags = mask_and_boxes(heat, threshold)
    flags = rec.get("alignment_flags", []) + flags
    prompt = review.get("prompt_short", rec["prompt_short"])
    if not boxes or not prompt.strip():
        review_rows.append(dict(id=rec["id"], status="skipped", flags=";".join(flags + ["check_prompt_and_boxes"])))
        continue
    uid = rec["id"]
    for folder in ("original", "result", "overlay", "masks", "heatmaps", "previews"):
        (export_dir / folder).mkdir(exist_ok=True)
    Image.fromarray(a).save(export_dir / "original" / f"{uid}.png")
    Image.fromarray(b).save(export_dir / "result" / f"{uid}.png")
    Image.fromarray(mask).save(export_dir / "masks" / f"{uid}.png")
    np.save(export_dir / "heatmaps" / f"{uid}.npy", heat)
    split_number = int(hashlib.sha256(rec["source_url"].encode()).hexdigest()[:8], 16) % 100
    split = "train" if split_number < 80 else "validation" if split_number < 90 else "test"
    for variant in range(N_VARIANTS):
        variant_seed = (SEED + int(uid[:8], 16) + variant) % 2**32
        variant_id = f"{uid}_v{variant}"
        style = style_for(rec, variant)
        overlay = draw_scribble(a, boxes, variant_seed, style, mask)
        overlay.save(export_dir / "overlay" / f"{variant_id}.png")
        row = dict(id=variant_id, source_pair_id=uid, operation=rec["operation"], split=split,
            prompt_short=prompt, prompt_original=rec["prompt_original"],
            original_file_name=f"original/{uid}.png", overlay_file_name=f"overlay/{variant_id}.png",
            result_file_name=f"result/{uid}.png", mask_file_name=f"masks/{uid}.png",
            boxes=boxes, seed=variant_seed, scribble_style=style, scribble_version=SCRIBBLE_VERSION, method=EXPORT_METHOD, threshold=threshold,
            review_status=status, flags=flags, alignment={k: rec.get(k) for k in ("source_size", "result_size", "aspect_ratio_error")}, source_url=rec["source_url"], result_url=rec["result_url"])
        rows.append(row)
    preview = Image.new("RGB", (a.shape[1]*3, a.shape[0]))
    for i, image in enumerate((Image.fromarray(a), draw_scribble(a, boxes, SEED, style_for(rec, 0), mask), Image.fromarray(b))):
        preview.paste(image, (i*a.shape[1], 0))
    preview.thumbnail((1800, 700))
    preview.save(export_dir / "previews" / f"{uid}.png")
    review_rows.append(dict(id=uid, status=status, flags=";".join(flags)))

(export_dir / "metadata.jsonl").write_text(
    "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows), encoding="utf-8")
with (export_dir / "review.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "status", "flags"])
    writer.writeheader(); writer.writerows(review_rows)
write_json(export_dir / "run_config.json", dict(**export_config, selection=selection_config,
    packages=subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True).splitlines()))
write_json(export_dir / "reviews.json", reviews)
(export_dir / "README.txt").write_text(
    "Pico-Banana derived research preview. Check source dataset terms before reuse.\n"
    "Masks are pseudo-labels. Only review_status=approved has been manually accepted.\n"
    "original + overlay are conditioning inputs; result is target, not a third conditioning image.\n",
    encoding="utf-8")
for row in rows:
    for key in ("original_file_name", "overlay_file_name", "result_file_name", "mask_file_name"):
        assert (export_dir / row[key]).is_file()
assert all(len({r["split"] for r in rows if r["source_url"] == source}) == 1
           for source in {r["source_url"] for r in rows})
archive = shutil.make_archive(str(export_dir), "zip", root_dir=export_dir)
print("Exported rows:", len(rows), "pairs:", len({r["source_pair_id"] for r in rows}))
print("Review:", dict(Counter(r["status"] for r in review_rows)))
print("ZIP:", archive)
display(FileLink(archive))


In [ ]:
# Chạy lại cell này nếu muốn tải ZIP về máy; không cần Drive.
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    display(FileLink(archive))


## Giới hạn và bước tiếp theo
- Chỉ xử lý edit cục bộ. Cặp lệch bố cục, toàn ảnh đổi style, nhiều component hoặc
  bóng/phản chiếu lớn cần duyệt; kiểm tra tỷ lệ không phát hiện được mọi sai lệch hình học.
- Upload ZIP không cần Flickr; tải model vẫn cần kết nối mạng.
- SIGMA visual dùng fallback của checkpoint có nhánh text, không phải mô hình được
  huấn luyện riêng chỉ với hai ảnh. So sánh pixel chỉ là baseline, không phải nhãn chuẩn.
- Duyệt prompt rút gọn có sẵn: một số câu có thể là mô tả kết quả thay vì yêu cầu chỉnh sửa.
- Nét tổng hợp là box rung nhẹ; cần kiểm tra thêm với scribble thật trước khi scale.
- Trainer FLUX.2 phải hỗ trợ hai ảnh điều kiện theo thứ tự cố định: original, overlay.
  Result chỉ làm target. Chưa có bước train trong notebook này.
- Nhánh text dùng thêm Qwen/GroundingDINO/SAM2 nên tốn VRAM hơn; nếu OOM, thử ít mẫu,
  giảm INFER_SIZE xuống bội số 14 hoặc dùng GPU nhiều VRAM hơn. Ít mẫu giảm thời gian,
  không giảm peak VRAM của một lần inference. Chưa xác nhận end-to-end trên GPU Colab.

- Đánh giá 8 cặp: visual/text tương đương về vùng chính; text có thêm đốm ở áo khoác
  và bảng biểu tình. Mẫu kính thiên văn→cây có mask lan nền; mẫu tượng sai instruction
  ngay từ result (xóa cả dãy). Cần loại/sửa target, đa dạng nét không sửa được nhãn sai.
- Đa dạng scribble bằng kiểu nét và seed; luôn giữ các biến thể cùng cặp trong một split.
